In [ ]:
from google.colab import files
upload=files.upload()

Saving Groundwater_dataset.csv to Groundwater_dataset.csv


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay


In [ ]:
df=pd.read_csv("Groundwater_dataset.csv")

In [ ]:
df.head()

,LAT,LONG,WL_MBGL,elevation,potential,slope,soil_subsurface,soil_surface,tpi
0,30.47000,73.08917,6.127041,157,1,0.000000,7,7,-0.889590
1,30.61306,73.00361,7.460307,170,1,1.077566,7,7,3.309148
2,30.30500,72.80083,7.578483,152,1,2.147576,7,7,1.037855
3,30.27167,72.85556,8.816340,150,1,2.142427,7,7,-0.138801
4,30.36000,72.99472,10.818571,156,1,0.927410,7,7,-0.441640


In [ ]:
X=df.drop(["potential","LAT","LONG","WL_MBGL"],axis=1)

In [ ]:
y=df["potential"]

In [ ]:
X_train, X_test, y_train, y_test= train_test_split(X,y,test_size=0.2, random_state=42)

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
lr_model=LogisticRegression(random_state=42)

In [ ]:
lr_model.fit(X_train,y_train)
lr_pred=lr_model.predict(X_test)
print(accuracy_score(y_test,lr_pred))
print(classification_report(y_test,lr_pred))

0.5808823529411765
              precision    recall  f1-score   support

           0       0.51      0.39      0.44       230
           1       0.62      0.72      0.67       314

    accuracy                           0.58       544
   macro avg       0.56      0.55      0.55       544
weighted avg       0.57      0.58      0.57       544



In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
fr_model=RandomForestClassifier(random_state=42)

In [ ]:
fr_model.fit(X_train,y_train)
frpred=fr_model.predict(X_test)
print(accuracy_score(y_test,frpred))
print(classification_report(y_test,frpred))

0.6194852941176471
              precision    recall  f1-score   support

           0       0.56      0.50      0.53       230
           1       0.66      0.70      0.68       314

    accuracy                           0.62       544
   macro avg       0.61      0.60      0.60       544
weighted avg       0.62      0.62      0.62       544



In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
dtmodel=DecisionTreeClassifier(random_state=42)

In [ ]:
dtmodel.fit(X_train,y_train)
dtpred=dtmodel.predict(X_test)
print(accuracy_score(y_test,dtpred))

0.5772058823529411


In [ ]:
print(classification_report(y_test,dtpred))

              precision    recall  f1-score   support

           0       0.50      0.54      0.52       230
           1       0.64      0.60      0.62       314

    accuracy                           0.58       544
   macro avg       0.57      0.57      0.57       544
weighted avg       0.58      0.58      0.58       544



In [ ]:
importances=fr_model.feature_importances_

In [ ]:
print(importances)

[0.30609075 0.28770725 0.047761   0.03556359 0.3228774 ]


In [ ]:
X.columns

Index(['elevation', 'slope', 'soil_subsurface', 'soil_surface', 'tpi'], dtype='object')

In [ ]:
param_grid = {
    'n_estimators': [150,250],
    'max_depth': [6, 8, 10, 12],
    'min_samples_leaf': [1,2,4],
    #'class_weight':[{0: 1.1, 1: 1.0}, {0: 1.2, 1: 1.0}, {0: 1.3, 1: 1.0}],
    'criterion': ['entropy']
}

In [ ]:
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring='precision',
    n_jobs=-1,
    verbose=1
)

In [ ]:
grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_


Fitting 3 folds for each of 24 candidates, totalling 72 fits


In [ ]:
print(grid_search.best_params_)

{'criterion': 'entropy', 'max_depth': 10, 'min_samples_leaf': 1, 'n_estimators': 150}


In [ ]:
predictions = best_model.predict(X_test)
#predictions = best_model.predict_proba(X_test)[:,1]

In [ ]:
#y_pred_adjusted = (predictions>= 0.55).astype(int)

In [ ]:
print(classification_report(y_test,predictions))

              precision    recall  f1-score   support

           0       0.63      0.43      0.51       230
           1       0.66      0.82      0.73       314

    accuracy                           0.65       544
   macro avg       0.65      0.62      0.62       544
weighted avg       0.65      0.65      0.64       544



In [ ]:
import joblib

In [ ]:
joblib.dump(best_model,"groundwater_rf_model.joblib")

['groundwater_rf_model.joblib']

In [ ]:
#load=joblib.load("groundwater_rf_model.joblib")

In [ ]:
import sys
print(f"Pandas:       {pd.__version__}")
print(f"NumPy:        {np.__version__}")
print(f"Scikit-Learn: {sys.modules['sklearn'].__version__}")
print(f"Joblib: {joblib.__version__}")

Pandas:       2.2.2
NumPy:        2.0.2
Scikit-Learn: 1.6.1
Joblib: 1.5.3
